# Predicting Monetary Policy Stance from FOMC Minutes

CS 5100
August 12, 2026
Jinyu Chen and Dylan Ullrich

This notebook documents the reproducible workflow for classifying sentence-level policy stance, aggregating meeting-level tone, evaluating the dictionary and weakly supervised FinBERT classifiers against a human audit, and examining exploratory associations with Treasury yields.

## 1. Data Sources

- FOMC minutes: official public Federal Reserve meeting calendar and historical pages.
- Market data: FRED `DGS10` and `DGS2` Treasury Constant Maturity Rate CSV files.

Run the following cell only if raw data have not already been downloaded.

In [ ]:
# Optional: download official public data.
# !python3 ../scripts/fetch_data.py --start-year 2008 --end-year 2026

## 2. Sentence-Level Tone Classifier

The baseline classifier is intentionally transparent. A sentence is labeled hawkish when hawkish monetary-policy terms outnumber dovish terms, dovish when the reverse is true, and neutral otherwise. The document score is:

`net hawkish score = (hawkish sentence count - dovish sentence count) / total sentence count`

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path('..').resolve()
sys.path.insert(0, str(PROJECT_ROOT / 'scripts'))

import pandas as pd
from IPython.display import Image, display
from analyze_tone import HAWKISH_TERMS, DOVISH_TERMS, score_sentence

print('Sample hawkish terms:', sorted(list(HAWKISH_TERMS))[:8])
print('Sample dovish terms:', sorted(list(DOVISH_TERMS))[:8])

In [ ]:
examples = [
    'Inflation pressures remained elevated and several participants supported further tightening.',
    'The Committee judged that downside risks and labor market slack warranted continued accommodation.',
    'Participants reviewed recent data on household spending and business fixed investment.'
]

for sentence in examples:
    print(score_sentence(sentence), '::', sentence)

## 3. Run the Full Analysis

The script scores every downloaded minute, aligns FRED yields using the latest observation on or before each FOMC meeting date, computes correlations, and creates figures. The corresponding minutes are released later.

This is not a minutes-release event study; it describes associations between meeting-related policy tone and the yield environment around FOMC meeting dates.

In [ ]:
# Recompute scores and figures.
# !python3 ../scripts/analyze_tone.py

## 4. Processed Scores

The panel below contains one row per FOMC meeting: sentence counts, net hawkish score, smoothed score, and aligned Treasury yields.

In [ ]:
panel = pd.read_csv(PROJECT_ROOT / 'data/processed/fomc_tone_market_panel.csv')
panel.head()

In [ ]:
panel[['date', 'sentence_count', 'hawkish_sentences', 'dovish_sentences', 'net_hawkish_score', 'score_ma3', 'DGS10', 'DGS2']].tail()

## 5. Main Visualization

This chart compares the FOMC tone index with the 10-year Treasury yield and marks major policy regimes.

In [ ]:
display(Image(filename=str(PROJECT_ROOT / 'figures/tone_vs_dgs10.png')))

## 6. Exploratory Market Association

The table reports exploratory correlations between the three-meeting-smoothed tone index and `DGS10` and `DGS2` at lags zero through three meetings. These eight comparisons are descriptive rather than prespecified causal models. Because the smoothed observations overlap and both tone and yields are persistent, conventional Pearson p-values do not account for serial dependence or multiple-comparison selection. Yields are aligned to meeting dates while minutes are released later, so this is not a minutes-release event study. Market association is separate from classifier evaluation against the human audit.

In [ ]:
corr = pd.read_csv(PROJECT_ROOT / 'data/processed/correlation_results.csv')
corr

In [ ]:
display(Image(filename=str(PROJECT_ROOT / 'figures/tone_yield_scatter.png')))

## 7. FinBERT Comparison and Independent Evaluation

`scripts/finbert_optional.py` fine-tuned `ProsusAI/finbert` as a hawkish/dovish/neutral classifier using dictionary-generated weak labels. Teacher-label agreement is not independent accuracy. Both FinBERT and the dictionary baseline were therefore evaluated against the same blind 30-sentence human audit.

In [ ]:
metrics = pd.read_csv(PROJECT_ROOT / 'data/processed/classifier_metrics.csv')
metrics

In [ ]:
display(Image(filename=str(PROJECT_ROOT / 'figures/confusion_matrix_audit.png')))

In [ ]:
examples = pd.read_csv(PROJECT_ROOT / 'data/processed/classifier_examples.csv')
examples[['example_type', 'sentence', 'manual_label', 'dictionary_label', 'finbert_label']]

## 8. Conclusion

The project demonstrates an end-to-end NLP workflow using public data, a transparent dictionary baseline, a weakly supervised FinBERT comparison, independent human-audit evaluation, and exploratory market visualizations. The dictionary baseline scored 53.3% accuracy versus 50.0% for FinBERT on the 30-sentence audit; the audit is too small to establish a meaningful performance difference. Market association is not classifier ground truth.